# 04 CNN Model

This notebook builds and evaluates a simple 1D CNN for software effort prediction.

In [6]:
from importlib import import_module, util
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

drive = None
if util.find_spec("google.colab") is not None:
    drive = import_module("google.colab.drive")
    IN_COLAB = True
else:
    IN_COLAB = False


def resolve_project_root() -> Path:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        candidate_roots = [
            drive_root / "Software-cost-Estimation",
            drive_root / "Colab Notebooks" / "Software-cost-Estimation",
        ]
        for candidate in candidate_roots:
            if (candidate / "src").exists():
                return candidate

        for src_dir in drive_root.rglob("src"):
            if src_dir.is_dir() and src_dir.parent.name == "Software-cost-Estimation":
                return src_dir.parent

        raise FileNotFoundError(
            "Could not find the 'Software-cost-Estimation' project folder in Google Drive. "
            "Upload the full project folder to MyDrive or Colab Notebooks."
        )

    root_dir = Path.cwd()
    if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
        root_dir = root_dir.parent
    return root_dir


root_dir = resolve_project_root()
if not (root_dir / "src").exists():
    raise FileNotFoundError(f"Could not find project root containing 'src' directory. Checked: {root_dir}")

sys.path.insert(0, str(root_dir))
print("Using project root:", root_dir)

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import inverse_log_transform, log_transform_target

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col]).values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED
)

# Keep the final test split untouched until the very end of evaluation.
y_train_fit = log_transform_target(y_train, use_log_transform=True).astype(np.float32)
y_val_fit = log_transform_target(y_val, use_log_transform=True).astype(np.float32)

X_train_cnn = reshape_for_cnn(X_train)
X_val_cnn = reshape_for_cnn(X_val)
X_test_cnn = reshape_for_cnn(X_test)

print("Using dataset:", dataset_path.name)
print("Train/val/test shapes:", X_train_cnn.shape, X_val_cnn.shape, X_test_cnn.shape)

Using project root: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation
Using dataset: desharnais_processed.csv
Train/val/test shapes: (48, 12, 1) (16, 12, 1) (17, 12, 1)


In [7]:
model = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
)

model, history = train_cnn_model(
    model,
    X_train_cnn, y_train_fit,
    X_val_cnn, y_val_fit,
    epochs=50,
    batch_size=32,
    verbose=0,
)

print("Best validation MAE:", min(history["val_mean_absolute_error"]))

Best validation MAE: 6.926349639892578


In [8]:
cnn_preds_log = model.predict(X_test_cnn, verbose=0).ravel()
cnn_preds = inverse_log_transform(cnn_preds_log, use_log_transform=True)
cnn_metrics = evaluate_predictions("cnn_baseline", y_test, cnn_preds)
cnn_metrics

,model,mae,rmse,r2,mape,mdmre,pred25
0,cnn_baseline,4532.925978,5771.363476,-1.61065,99.84714,0.999059,0.0


In [9]:
cnn_metrics_path = save_metrics(
    cnn_metrics,
    file_name="cnn_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved CNN metrics to:", cnn_metrics_path)

Saved CNN metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\cnn_metrics.csv
